In [1]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.python.client import device_lib

print(device_lib.list_local_devices())
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 18091552327627020156
xla_global_id: -1
]


In [2]:
tf.test.is_gpu_available()
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


In [3]:
def DenseNet(x):
    k = 32
    compression = 0.5

    x = layers.Conv2D(k * 2, (7,7), strides = 2, padding = 'same', input_shape=(224,224,3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.MaxPool2D((3,3), strides = 2, padding = 'same')(x)

    for i in range(6):
        x_1 = layers.Conv2D(k * 4, (1,1), strides = 1, padding = 'same')(x)
        x_1 = layers.BatchNormalization()(x_1)
        x_1 = layers.Activation('relu')(x_1)

        x_1 = layers.Conv2D(k, (3,3), strides = 1, padding = 'same')(x_1)
        x_1 = layers.BatchNormalization()(x_1)
        x_1 = layers.Activation('relu')(x_1)

        x = layers.Concatenate()([x, x_1])
    current_shape = int(x.shape[-1])
    x = layers.Conv2D(int(current_shape * compression), (1,1), strides = 1, padding = 'same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.AveragePooling2D((2,2), strides = 2, padding = 'same')(x)

    for i in range(12):
        x_1 = layers.Conv2D(k * 4, (1,1), strides = 1, padding = 'same')(x)
        x_1 = layers.BatchNormalization()(x_1)
        x_1 = layers.Activation('relu')(x_1)

        x_1 = layers.Conv2D(k, (3,3), strides = 1, padding = 'same')(x_1)
        x_1 = layers.BatchNormalization()(x_1)
        x_1 = layers.Activation('relu')(x_1)

        x = layers.Concatenate()([x, x_1])

    current_shape = int(x.shape[-1])
    x = layers.Conv2D(int(current_shape * compression), (1,1), strides = 1, padding = 'same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.AveragePooling2D((2,2), strides = 2, padding = 'same')(x)

    for i in range(24):
        x_1 = layers.Conv2D(k * 4, (1,1), strides = 1, padding = 'same')(x)
        x_1 = layers.BatchNormalization()(x_1)
        x_1 = layers.Activation('relu')(x_1)

        x_1 = layers.Conv2D(k, (3,3), strides = 1, padding = 'same')(x_1)
        x_1 = layers.BatchNormalization()(x_1)
        x_1 = layers.Activation('relu')(x_1)

        x = layers.Concatenate()([x, x_1])

    current_shape = int(x.shape[-1])
    x = layers.Conv2D(int(current_shape * compression), (1,1), strides = 1, padding = 'same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.AveragePooling2D((2,2), strides = 2, padding = 'same')(x)

    for i in range(16):
        x_1 = layers.Conv2D(k * 4, (1,1), strides = 1, padding = 'same')(x)
        x_1 = layers.BatchNormalization()(x_1)
        x_1 = layers.Activation('relu')(x_1)

        x_1 = layers.Conv2D(k, (3,3), strides = 1, padding = 'same')(x_1)
        x_1 = layers.BatchNormalization()(x_1)
        x_1 = layers.Activation('relu')(x_1)

        x = layers.Concatenate()([x, x_1])

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(1, activation = 'sigmoid')(x)

    return x

In [4]:
batch_size = 32
epoch = 50
learning_rate = 0.001

dataset_path = os.path.join('./file/Image')
train_dataset_path = dataset_path + '/train'
train_data_generator = ImageDataGenerator(rescale = 1./255)
train_dataset = train_data_generator.flow_from_directory(train_dataset_path,
                                                         shuffle = True,
                                                         target_size=(224,224),
                                                         batch_size=batch_size,
                                                         class_mode='categorical')
valid_dataset_path = dataset_path + "/valid"
valid_data_generator = ImageDataGenerator(rescale = 1./255)
valid_dataset = valid_data_generator.flow_from_directory(valid_dataset_path,
                                                         shuffle = True,
                                                         target_size=(224,224),
                                                         batch_size=batch_size,
                                                         class_mode='categorical')

Found 378 images belonging to 1 classes.
Found 79 images belonging to 1 classes.


In [5]:
input_shape = layers.Input(shape=(224,224,3), dtype = 'float32', name = 'input')
output = DenseNet(input_shape)
model = tf.keras.Model(input_shape, output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

C:\Users\user\.conda\envs\Torch\lib\site-packages\keras\src\layers\convolutional\base_conv.py:99: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)            │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 112, 112, 64)      │           9,472 │ input[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 112, 112, 64)      │             256 │ conv2d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation (Activation)       │ (None, 112, 112, 64)      │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 56, 56, 64)        │               0 │ activation[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 56, 56, 128)       │           8,320 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 56, 56, 128)       │             512 │ conv2d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_1 (Activation)     │ (None, 56, 56, 128)       │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 56, 56, 32)        │          36,896 │ activation_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 56, 56, 32)        │             128 │ conv2d_2[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_2 (Activation)     │ (None, 56, 56, 32)        │               0 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 56, 56, 96)        │               0 │ max_pooling2d[0][0],       │
│                               │                           │                 │ activation_2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 56, 56, 128)       │          12,416 │ concatenate[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 56, 56, 128)       │             512 │ conv2d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_3 (Activation)     │ (None, 56, 56, 128)       │               

 Total params: 6,922,433 (26.41 MB)

 Trainable params: 6,901,953 (26.33 MB)

 Non-trainable params: 20,480 (80.00 KB)